<a href="https://colab.research.google.com/github/HarithaGottumukkala/DATA266-1598_hw3/blob/main/Homework3_1598.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA 266 — Homework 3
## Prompt Engineering, Self-Attention, and Causal Masking

### Personal Parameters

| Parameter | Value |
|-----------|-------|
| SID4 | 1598 |
| SEED | 1598 |
| SLICE | 598 |
| HP_ID | 2 |
| CLS_A | 8 |
| CLS_B | 5 |

In this assignment, I will compare six prompt engineering techniques
using two examples for each technique.

I will also implement and train two single-head attention models on
the supplied text. One model will use unmasked attention, and the
other will use causal attention. I will compare their training results
and visualize their learned attention weights.

The unmasked and causal models are the two required training
experiments for this assignment.

# 0. Setup and Reproducibility

## 0.1 Installing the Required Packages

I will use PyTorch to build the attention models, NumPy for numerical
operations, and Matplotlib for the plots.

For the prompt experiments, I will use the langchain-ollama package
to connect my Python code to Ollama.

The following cell installs these packages if they are not already
available in the runtime.

In [3]:
%pip install -q numpy matplotlib torch langchain-ollama

## 0.2 Personal Parameters and Random Seeds

I will calculate my personal parameters using the formulas from the
standing requirements.

The attention models will start with randomly initialized weights.
I will use SEED = 1598 for Python, NumPy, and PyTorch to help reproduce
the experiments under the same conditions.

I will train the small attention models on the CPU. Ollama manages
the hardware used by the separate language model for Question 1.

This assignment does not provide an HP_ID configuration or ask for a
dataset slice or focus classes. I will report those parameters but
use the complete supplied text.

In [4]:
import platform
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn


# Personal parameters.
SID4 = 1598

SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6

CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10


def set_seed(seed):
    """Reset the random seeds before an experiment."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

# This device is for our from-scratch attention models.
device = torch.device("cpu")

torch.use_deterministic_algorithms(True)

personal_parameters = {
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
}

for name, value in personal_parameters.items():
    print(f"{name} = {value}")

print("\nAttention model device:", device)
print("Python version:", platform.python_version())
print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print(
    "Deterministic algorithms enabled:",
    torch.are_deterministic_algorithms_enabled(),
)

SID4 = 1598
SEED = 1598
SLICE = 598
HP_ID = 2
CLS_A = 8
CLS_B = 5

Attention model device: cpu
Python version: 3.13.15
NumPy version: 2.1.3
PyTorch version: 2.11.0+cpu
Deterministic algorithms enabled: True


## 0.3 Recording the Run Output

I will use a function called log() to display experiment messages
in the notebook and save the same text in RUN_LOG.txt.

I will use it for the prompts, model responses, training results,
and other measurements. The log will also record the environment
details and any notebook execution errors.

The function does not replace the notebook's output streams.
Ordinary print() statements will continue to display normally.

In [5]:
from datetime import datetime
from pathlib import Path
from importlib.metadata import version
import traceback

RUN_LOG_PATH = Path("RUN_LOG.txt")


def timestamp():
    """Return the current time, including its timezone."""
    return datetime.now().astimezone().isoformat(timespec="seconds")


def log(*values):
    """Display a message and append it to the run log."""
    message = " ".join(str(value) for value in values)

    print(message)

    with RUN_LOG_PATH.open("a", encoding="utf-8") as log_file:
        log_file.write(message + "\n")


# Record cell errors without replacing Colab's output streams.
ipython = get_ipython()

# Avoid registering the same callback twice when rerunning this cell.
if "_record_cell_error" in globals():
    try:
        ipython.events.unregister("post_run_cell", _record_cell_error)
    except ValueError:
        pass


def _record_cell_error(result):
    error = result.error_before_exec or result.error_in_exec

    if error is not None:
        details = "".join(
            traceback.format_exception(
                type(error), error, error.__traceback__
            )
        )
        log(f"\n[{timestamp()}] Cell error:")
        log(details)


ipython.events.register("post_run_cell", _record_cell_error)

# Record the environment for this run.
log(f"\nRun session started: {timestamp()}")

for name, value in personal_parameters.items():
    log(f"{name} = {value}")

log("Attention model device:", device)
log("Python version:", platform.python_version())
log("NumPy version:", np.__version__)
log("Matplotlib version:", version("matplotlib"))
log("PyTorch version:", torch.__version__)
log("LangChain Ollama version:", version("langchain-ollama"))
log("LangChain Core version:", version("langchain-core"))
log(
    "Deterministic algorithms enabled:",
    torch.are_deterministic_algorithms_enabled(),
)
log("Log location:", RUN_LOG_PATH.resolve())


Run session started: 2026-09-10T21:11:38+00:00
SID4 = 1598
SEED = 1598
SLICE = 598
HP_ID = 2
CLS_A = 8
CLS_B = 5
Attention model device: cpu
Python version: 3.13.15
NumPy version: 2.1.3
Matplotlib version: 3.10.0
PyTorch version: 2.11.0+cpu
LangChain Ollama version: 1.1.0
LangChain Core version: 1.6.1
Deterministic algorithms enabled: True
Log location: /content/RUN_LOG.txt


# 1. Prompt Engineering Experiments

I will compare the following six techniques:

1. Zero-shot
2. Few-shot
3. Chain-of-Thought
4. Zero-shot Chain-of-Thought
5. Meta-prompting
6. Tree of Thoughts

Each technique will have two separate code examples: one for a math
problem and one for a scheduling puzzle. This gives 12 examples in total.

I will reuse the same two tasks so that I can compare how changing
the prompt affects the response. I will keep the model and generation
settings consistent.

For each response, I will check whether the answer is correct,
whether it follows the instructions, and whether its explanation
is useful.

These experiments cover only two tasks, so they cannot establish
that one technique is always better than another.

## 1.1 Tasks and Reference Answers

### Example A — Math Word Problem

A student club buys 5 packs of notebooks, with 12 notebooks in each
pack. The club receives 8 additional notebooks as a donation and
then gives away 47 notebooks. The remaining notebooks are placed
into gift bags containing 3 notebooks each.

How many complete gift bags can the club make?

The reference answer is **7 gift bags**:

- Purchased notebooks: 5 × 12 = 60.
- After the donation: 60 + 8 = 68.
- Remaining notebooks: 68 − 47 = 21.
- Complete gift bags: 21 ÷ 3 = 7.

### Example B — Scheduling Puzzle

Four students—Riya, Omar, Mei, and Luis—must present one at a time.
Each student presents exactly once.

The rules are:

- Luis presents first.
- Riya presents before Omar.
- Mei presents immediately after Riya.

What is the presentation order?

The reference answer is **Luis, Riya, Mei, Omar**.

Luis takes the first position. Riya and Mei must occupy consecutive
positions, in that order, and Omar must come after Riya. This places
Riya second, Mei third, and Omar fourth.

I will keep these reference answers separate from the prompts sent
to the model.

In [6]:
tasks = {
    "math": (
        "A student club buys 5 packs of notebooks, with 12 notebooks "
        "in each pack. The club receives 8 additional notebooks as a "
        "donation and then gives away 47 notebooks. The remaining "
        "notebooks are placed into gift bags containing 3 notebooks "
        "each. How many complete gift bags can the club make?"
    ),
    "logic": (
        "Four students—Riya, Omar, Mei, and Luis—must present one at "
        "a time. Each student presents exactly once. Luis presents "
        "first. Riya presents before Omar. Mei presents immediately "
        "after Riya. What is the presentation order?"
    ),
}

# These are for evaluation, not for inclusion in the model prompts.
reference_answers = {
    "math": 7,
    "logic": ["Luis", "Riya", "Mei", "Omar"],
}

for task_name, question in tasks.items():
    log(f"\n{task_name.upper()} TASK")
    log(question)


MATH TASK
A student club buys 5 packs of notebooks, with 12 notebooks in each pack. The club receives 8 additional notebooks as a donation and then gives away 47 notebooks. The remaining notebooks are placed into gift bags containing 3 notebooks each. How many complete gift bags can the club make?

LOGIC TASK
Four students—Riya, Omar, Mei, and Luis—must present one at a time. Each student presents exactly once. Luis presents first. Riya presents before Omar. Mei presents immediately after Riya. What is the presentation order?


## 1.2 Setting Up the Language Model

I will use qwen2.5:1.5b through Ollama for the prompt experiments.

Ollama will run the model inside my Colab runtime. LangChain will
send prompts to Ollama and return the model's responses to the notebook.

This is a pretrained model used only for Question 1. In Question 2,
I will implement and train the attention models myself using basic
PyTorch layers and tensor operations.

### 1.2.1 Installing Ollama

I have already installed the Python integration, langchain-ollama.
The next cell installs the Ollama application.

The cell will reuse an existing installation if one is available.
Its console output will also be appended to RUN_LOG.txt.

In [7]:
%%bash
set -euo pipefail

{
    echo "Starting Ollama installation check."

    if command -v ollama >/dev/null 2>&1; then
        echo "Ollama is already installed."
    else
        apt-get update -qq
        apt-get install -y -qq zstd

        curl -fsSL https://ollama.com/install.sh | sh
    fi

    echo "Ollama executable:"
    command -v ollama

} 2>&1 | tee -a RUN_LOG.txt

Starting Ollama installation check.
Ollama is already installed.
Ollama executable:
/usr/local/bin/ollama


### 1.2.2 Starting the Ollama Server

Ollama must be running before it can receive requests from Python.

I will check whether the server is already available. If it is not,
I will start it in the background and wait for it to respond.

The server will be accessible within this Colab runtime. Its
background messages will be saved in OLLAMA_SERVER.log.

In [8]:
import os
import json
import subprocess
import time

from urllib.request import urlopen
from urllib.error import URLError

OLLAMA_URL = "http://127.0.0.1:11434"
SERVER_LOG_PATH = Path("OLLAMA_SERVER.log")


def ollama_is_ready():
    """Return True when the Ollama server responds."""
    try:
        with urlopen(f"{OLLAMA_URL}/api/version", timeout=2) as response:
            return response.status == 200
    except (URLError, OSError):
        return False


if ollama_is_ready():
    log("Ollama is already running.")
else:
    log("Starting Ollama.")

    server_env = os.environ.copy()
    server_env["OLLAMA_HOST"] = "127.0.0.1:11434"

    with SERVER_LOG_PATH.open("a", encoding="utf-8") as server_log:
        ollama_process = subprocess.Popen(
            ["ollama", "serve"],
            stdout=server_log,
            stderr=subprocess.STDOUT,
            env=server_env,
        )

    for attempt in range(30):
        if ollama_is_ready():
            break
        time.sleep(1)


if not ollama_is_ready():
    if SERVER_LOG_PATH.exists():
        log(SERVER_LOG_PATH.read_text(encoding="utf-8")[-3000:])

    raise RuntimeError(
        "Ollama did not become ready. Check the server messages above."
    )


with urlopen(f"{OLLAMA_URL}/api/version", timeout=5) as response:
    server_info = json.load(response)

log("Ollama is ready.")
log("Ollama version:", server_info["version"])

Starting Ollama.
Ollama is ready.
Ollama version: 0.34.0


### 1.2.3 Downloading the Model

I will download qwen2.5:1.5b into the Colab runtime.

Downloading the model makes its pretrained weights available to
Ollama. It does not train or modify those weights.

I will record the model name and digest. The digest identifies
the downloaded model files used for the experiments.

In [9]:
from ollama import Client

MODEL_NAME = "qwen2.5:1.5b"

ollama_client = Client(host=OLLAMA_URL)

log(f"\n[{timestamp()}] Preparing model: {MODEL_NAME}")

previous_status = None

# Report each download stage once.
for update in ollama_client.pull(MODEL_NAME, stream=True):
    if update.status != previous_status:
        log(update.status)
        previous_status = update.status

# Confirm that the requested model is available.
downloaded_models = ollama_client.list().models

model_info = next(
    (
        item
        for item in downloaded_models
        if item.model == MODEL_NAME
    ),
    None,
)

if model_info is None:
    raise RuntimeError("The requested model was not found after downloading.")

MODEL_DIGEST = model_info.digest

log("Model is available.")
log("Model name:", MODEL_NAME)
log("Model digest:", MODEL_DIGEST)


[2026-09-10T21:13:34+00:00] Preparing model: qwen2.5:1.5b
pulling manifest
pulling 183715c43589
pulling 66b9ea09bd5b
pulling eb4402837c78
pulling 832dd9e00a68
pulling 377ac4d7aeef
verifying sha256 digest
writing manifest
success
Model is available.
Model name: qwen2.5:1.5b
Model digest: 65ec06548149b04c096a120e4a6da9d4017ea809c91734ea5631e89f96ddc57b


### 1.2.4 Connecting Ollama to LangChain

I will use LangChain's ChatOllama class to call the model.

The prompt experiments will use these settings:

| Setting | Value | Purpose |
|---------|-------|---------|
| Model | qwen2.5:1.5b | Use the same model for every technique |
| Temperature | 0 | Reduce randomness in the responses |
| Seed | 1598 | Use my assigned seed |
| Maximum generated tokens | 512 | Limit each response's length |
| Context window | 4096 tokens | Set the space available for the prompt and response |

Using fixed settings helps make the comparison consistent. Exact
responses can still depend on software versions and hardware.

I will first send a short test prompt to confirm that the connection
works. This setup check is separate from the 12 required examples.

In [10]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_URL,
    temperature=0,
    seed=SEED,
    num_predict=512,
    num_ctx=4096,
    validate_model_on_init=True,
)

log("\nPrompt experiment settings:")
log("Model:", MODEL_NAME)
log("Model digest:", MODEL_DIGEST)
log("Temperature:", llm.temperature)
log("Seed:", llm.seed)
log("Maximum generated tokens:", llm.num_predict)
log("Context window:", llm.num_ctx)

# A small connection check, separate from the graded examples.
test_prompt = "Reply with only the word: Ready"

log(f"\n[{timestamp()}] Starting connection test.")
log("Test prompt:", test_prompt)

test_response = llm.invoke(test_prompt)

log("Model response:", test_response.content)
log("Response completion reason:", test_response.response_metadata.get("done_reason"))
log(f"Connection test finished: {timestamp()}")


Prompt experiment settings:
Model: qwen2.5:1.5b
Model digest: 65ec06548149b04c096a120e4a6da9d4017ea809c91734ea5631e89f96ddc57b
Temperature: 0.0
Seed: 1598
Maximum generated tokens: 512
Context window: 4096

[2026-09-10T21:14:23+00:00] Starting connection test.
Test prompt: Reply with only the word: Ready
Model response: Ready
Response completion reason: stop
Connection test finished: 2026-09-10T21:14:35+00:00


## 1.3 Zero-Shot Prompting

Zero-shot prompting gives the model a task without showing any
completed examples.

For these experiments, I will send the math problem and scheduling
puzzle directly to the model. I will not provide demonstrations or
instructions to reason step by step.

I will ask the model to put its final answer on a clearly labeled
line. This will make the answer easier to identify when comparing
the responses.

Before running the examples, I will define a helper function that
displays and saves each prompt and response.

In [11]:
# Preserve existing results if this helper cell is rerun.
if "prompt_results" not in globals():
    prompt_results = {}

PROMPT_RESULTS_PATH = Path("prompt_results.json")


def run_prompt(technique, task_name, prompt, stage="final"):
    """Call the model and save the prompt, response, and settings."""
    started_at = timestamp()

    log(f"\n[{started_at}] {technique} | {task_name} | {stage}")
    log("\nPROMPT:")
    log(prompt)

    start_time = time.perf_counter()
    response = llm.invoke(prompt)
    elapsed_seconds = time.perf_counter() - start_time

    answer = response.content
    completion_reason = response.response_metadata.get("done_reason")

    log("\nMODEL RESPONSE:")
    log(answer)
    log("\nCompletion reason:", completion_reason)
    log("Elapsed time (seconds):", round(elapsed_seconds, 2))

    if completion_reason == "length":
        log("The response reached the token limit and may be incomplete.")

    result_key = f"{technique}:{task_name}:{stage}"

    prompt_results[result_key] = {
        "technique": technique,
        "task": task_name,
        "stage": stage,
        "timestamp": started_at,
        "model": MODEL_NAME,
        "model_digest": MODEL_DIGEST,
        "temperature": llm.temperature,
        "seed": llm.seed,
        "num_predict": llm.num_predict,
        "num_ctx": llm.num_ctx,
        "prompt": prompt,
        "response": answer,
        "completion_reason": completion_reason,
        "elapsed_seconds": elapsed_seconds,
        "usage": response.usage_metadata,
    }

    with PROMPT_RESULTS_PATH.open("w", encoding="utf-8") as results_file:
        json.dump(
            prompt_results,
            results_file,
            indent=2,
            ensure_ascii=False,
        )

    return answer


log("The prompt experiment helper is ready.")

The prompt experiment helper is ready.


### 1.3.1 Example A — Math Word Problem

I will give the model the notebook-and-gift-bag problem without
any worked examples.

The prompt will request a final answer in a consistent format.
It will not include the reference answer or suggest the calculations
needed to solve the problem.

In [12]:
zero_shot_math_prompt = (
    tasks["math"]
    + "\n\nEnd your response with: Final answer: <number> gift bags."
)

zero_shot_math_response = run_prompt(
    technique="zero_shot",
    task_name="math",
    prompt=zero_shot_math_prompt,
)

# Display the reference only after the model has answered.
log("\nReference answer:", reference_answers["math"], "gift bags")


[2026-09-10T21:16:25+00:00] zero_shot | math | final

PROMPT:
A student club buys 5 packs of notebooks, with 12 notebooks in each pack. The club receives 8 additional notebooks as a donation and then gives away 47 notebooks. The remaining notebooks are placed into gift bags containing 3 notebooks each. How many complete gift bags can the club make?

End your response with: Final answer: <number> gift bags.

MODEL RESPONSE:
First, let's calculate the total number of notebooks the club has:

- The club buys 5 packs of notebooks, with 12 notebooks in each pack.
  \[
  5 \text{ packs} \times 12 \text{ notebooks per pack} = 60 \text{ notebooks}
  \]

- The club also receives 8 additional notebooks as a donation.
  \[
  60 \text{ notebooks} + 8 \text{ notebooks} = 68 \text{ notebooks}
  \]

- The club gives away 47 notebooks.
  \[
  68 \text{ notebooks} - 47 \text{ notebooks} = 21 \text{ notebooks}
  \]

Next, we need to determine how many complete gift bags can be made with the remaining 2

#### Observation — Zero-Shot Math Example

The model correctly calculated that the club could make 7 gift bags.
It accounted for the purchased notebooks, the donation, and the
notebooks given away before dividing the remainder into groups of 3.

Although I did not request a step-by-step explanation, the model
provided the intermediate calculations on its own.

The response did not follow the exact final-line format I requested.
It ended with a boxed 7 instead of "Final answer: 7 gift bags."
The answer was correct, but the formatting instruction was not
followed exactly.

The call took 43.55 seconds in this run. This is a single observed
response time, so I will not treat it as a reliable speed benchmark.

### 1.3.2 Example B — Scheduling Puzzle

I will give the model the presentation rules without any solved
examples or instructions to reason step by step.

I will check whether the answer includes each student exactly once,
places Luis first, places Riya before Omar, and places Mei immediately
after Riya.

In [13]:
zero_shot_logic_prompt = (
    tasks["logic"]
    + "\n\nEnd your response with: "
    + "Final answer: <names in presentation order, separated by commas>."
)

zero_shot_logic_response = run_prompt(
    technique="zero_shot",
    task_name="logic",
    prompt=zero_shot_logic_prompt,
)

log(
    "\nReference answer:",
    ", ".join(reference_answers["logic"]),
)


[2026-09-10T21:18:32+00:00] zero_shot | logic | final

PROMPT:
Four students—Riya, Omar, Mei, and Luis—must present one at a time. Each student presents exactly once. Luis presents first. Riya presents before Omar. Mei presents immediately after Riya. What is the presentation order?

End your response with: Final answer: <names in presentation order, separated by commas>.

MODEL RESPONSE:
To determine the presentation order, let's analyze the given conditions step by step:

1. Luis presents first.
2. Riya presents before Omar.
3. Mei presents immediately after Riya.

Given these conditions, we can deduce the following:

- Since Luis presents first, the order starts with "Luis".
- Riya must present before Omar, so Riya cannot present second. Therefore, Riya must present third.
- Mei presents immediately after Riya, so Mei must present fourth.

Thus, the presentation order is: Luis, Riya, Mei, Omar.

Final answer: Luis, Riya, Mei, Omar.

Completion reason: stop
Elapsed time (seconds): 2

## 1.4 Few-Shot Prompting

Few-shot prompting gives the model a few completed examples before
asking it to solve a new task.

For each experiment, I will provide two demonstrations containing
a question and its answer. I will then ask the same target question
used in the zero-shot experiment.

The demonstrations will use different numbers or student names.
They will not include the answer to the target question.

I will compare the responses with the zero-shot results to check
whether the examples affect correctness, formatting, or the amount
of explanation.

### 1.4.1 Example A — Math Word Problem

I will show the model two solved notebook-and-gift-bag problems
before giving it the original problem.

The demonstrations include only the final answers. I will check
whether the model follows the demonstrated answer format and
correctly solves the new problem.

In [14]:
few_shot_math_prompt = """
Use the following examples as a guide to answer the new question.

Example 1:
Question: A club buys 2 packs of notebooks with 10 notebooks in each
pack. It receives 7 extra notebooks and gives away 15 notebooks.
The remaining notebooks are put into gift bags containing 4 each.
How many complete gift bags can it make?

Final answer: 3 gift bags.

Example 2:
Question: A club buys 4 packs of notebooks with 9 notebooks in each
pack. It receives 6 extra notebooks and gives away 18 notebooks.
The remaining notebooks are put into gift bags containing 6 each.
How many complete gift bags can it make?

Final answer: 4 gift bags.

New question:
""".strip()

few_shot_math_prompt += (
    "\n" + tasks["math"]
    + "\n\nEnd your response with: Final answer: <number> gift bags."
)

few_shot_math_response = run_prompt(
    technique="few_shot",
    task_name="math",
    prompt=few_shot_math_prompt,
)

log("\nReference answer:", reference_answers["math"], "gift bags")


[2026-09-10T21:19:59+00:00] few_shot | math | final

PROMPT:
Use the following examples as a guide to answer the new question.

Example 1:
Question: A club buys 2 packs of notebooks with 10 notebooks in each
pack. It receives 7 extra notebooks and gives away 15 notebooks.
The remaining notebooks are put into gift bags containing 4 each.
How many complete gift bags can it make?

Final answer: 3 gift bags.

Example 2:
Question: A club buys 4 packs of notebooks with 9 notebooks in each
pack. It receives 6 extra notebooks and gives away 18 notebooks.
The remaining notebooks are put into gift bags containing 6 each.
How many complete gift bags can it make?

Final answer: 4 gift bags.

New question:
A student club buys 5 packs of notebooks, with 12 notebooks in each pack. The club receives 8 additional notebooks as a donation and then gives away 47 notebooks. The remaining notebooks are placed into gift bags containing 3 notebooks each. How many complete gift bags can the club make?

End yo

### 1.4.2 Example B — Scheduling Puzzle

I will provide two solved scheduling examples before asking the
original question about Riya, Omar, Mei, and Luis.

The examples use different names and constraints. Their answers
show the expected format: a comma-separated presentation order.

I will check whether the model satisfies the original puzzle's
rules and follows the demonstrated format.

In [15]:
few_shot_logic_prompt = """
Use the following examples as a guide to answer the new question.

Example 1:
Question: Asha, Ben, Cora, and Dev each present exactly once.
Cora presents first. Asha presents before Dev.
Ben presents immediately after Asha.
What is the presentation order?

Final answer: Cora, Asha, Ben, Dev.

Example 2:
Question: Hana, Ivan, Jules, and Kiran each present exactly once.
Kiran presents last. Jules presents immediately before Hana.
Ivan presents before Jules.
What is the presentation order?

Final answer: Ivan, Jules, Hana, Kiran.

New question:
""".strip()

few_shot_logic_prompt += (
    "\n" + tasks["logic"]
    + "\n\nEnd your response with: "
    + "Final answer: <names in presentation order, separated by commas>."
)

few_shot_logic_response = run_prompt(
    technique="few_shot",
    task_name="logic",
    prompt=few_shot_logic_prompt,
)

log(
    "\nReference answer:",
    ", ".join(reference_answers["logic"]),
)


[2026-09-10T21:21:09+00:00] few_shot | logic | final

PROMPT:
Use the following examples as a guide to answer the new question.

Example 1:
Question: Asha, Ben, Cora, and Dev each present exactly once.
Cora presents first. Asha presents before Dev.
Ben presents immediately after Asha.
What is the presentation order?

Final answer: Cora, Asha, Ben, Dev.

Example 2:
Question: Hana, Ivan, Jules, and Kiran each present exactly once.
Kiran presents last. Jules presents immediately before Hana.
Ivan presents before Jules.
What is the presentation order?

Final answer: Ivan, Jules, Hana, Kiran.

New question:
Four students—Riya, Omar, Mei, and Luis—must present one at a time. Each student presents exactly once. Luis presents first. Riya presents before Omar. Mei presents immediately after Riya. What is the presentation order?

End your response with: Final answer: <names in presentation order, separated by commas>.

MODEL RESPONSE:
Final answer: Mei, Riya, Omar, Luis.

Completion reason: sto

#### Observation — Few-Shot Scheduling Example

The model returned "Final answer: Mei, Riya, Omar, Luis."
It followed the requested answer format, but the schedule was incorrect.

All four students appeared exactly once, and Riya appeared before
Omar. However, Luis was placed last instead of first, and Mei appeared
before Riya instead of immediately after her.

The correct order is Luis, Riya, Mei, Omar.

This example shows that a response can match the demonstrated format
without satisfying the task's rules. Providing two solved examples
was not enough for the model to solve this particular puzzle correctly.

The response ended normally with completion reason "stop", so the
incorrect answer was not caused by reaching the response-length limit.
The call took 13.57 seconds in this run.

## 1.5 Chain-of-Thought Prompting

For this experiment, I will provide worked examples that include
intermediate steps before the final answer.

I will use the same demonstration problems from the few-shot section,
but add their solution steps. The target questions and model settings
will remain the same.

I will check whether these worked examples help the model perform
the calculations and apply the scheduling constraints correctly.

A detailed explanation does not guarantee a correct answer, so I will
still check the final result against the reference answer.

### 1.5.1 Example A — Math Word Problem

The worked examples will show how to calculate the starting number
of notebooks, add donations, subtract the notebooks given away,
and divide the remainder into gift bags.

I will then ask the model to solve the original problem and provide
a brief explanation with its final answer.

In [16]:
cot_math_prompt = """
Use the worked examples below as a guide.
For the new question, explain your solution briefly.

Example 1:
Question: A club buys 2 packs of notebooks with 10 notebooks in each
pack. It receives 7 extra notebooks and gives away 15 notebooks.
The remaining notebooks are put into gift bags containing 4 each.
How many complete gift bags can it make?

Solution:
The club buys 2 × 10 = 20 notebooks.
After the donation, it has 20 + 7 = 27 notebooks.
After giving away 15, it has 27 - 15 = 12 notebooks.
Each bag holds 4 notebooks, so it can make 12 ÷ 4 = 3 bags.
Final answer: 3 gift bags.

Example 2:
Question: A club buys 4 packs of notebooks with 9 notebooks in each
pack. It receives 6 extra notebooks and gives away 18 notebooks.
The remaining notebooks are put into gift bags containing 6 each.
How many complete gift bags can it make?

Solution:
The club buys 4 × 9 = 36 notebooks.
After the donation, it has 36 + 6 = 42 notebooks.
After giving away 18, it has 42 - 18 = 24 notebooks.
Each bag holds 6 notebooks, so it can make 24 ÷ 6 = 4 bags.
Final answer: 4 gift bags.

New question:
""".strip()

cot_math_prompt += (
    "\n" + tasks["math"]
    + "\n\nEnd your response with: Final answer: <number> gift bags."
)

cot_math_response = run_prompt(
    technique="chain_of_thought",
    task_name="math",
    prompt=cot_math_prompt,
)

log("\nReference answer:", reference_answers["math"], "gift bags")


[2026-09-10T21:23:19+00:00] chain_of_thought | math | final

PROMPT:
Use the worked examples below as a guide.
For the new question, explain your solution briefly.

Example 1:
Question: A club buys 2 packs of notebooks with 10 notebooks in each
pack. It receives 7 extra notebooks and gives away 15 notebooks.
The remaining notebooks are put into gift bags containing 4 each.
How many complete gift bags can it make?

Solution:
The club buys 2 × 10 = 20 notebooks.
After the donation, it has 20 + 7 = 27 notebooks.
After giving away 15, it has 27 - 15 = 12 notebooks.
Each bag holds 4 notebooks, so it can make 12 ÷ 4 = 3 bags.
Final answer: 3 gift bags.

Example 2:
Question: A club buys 4 packs of notebooks with 9 notebooks in each
pack. It receives 6 extra notebooks and gives away 18 notebooks.
The remaining notebooks are put into gift bags containing 6 each.
How many complete gift bags can it make?

Solution:
The club buys 4 × 9 = 36 notebooks.
After the donation, it has 36 + 6 = 42 notebo

### 1.5.2 Example B — Scheduling Puzzle

The worked examples will show how to combine the presentation rules
to construct a valid order.

In the few-shot experiment, the model produced an incorrect schedule.
I will now check whether including solution steps helps it satisfy
the rules in the original puzzle.

I will evaluate the final order and check whether the explanation
agrees with that order.

In [17]:
cot_logic_prompt = """
Use the worked examples below as a guide.
For the new question, explain your solution briefly.

Example 1:
Question: Asha, Ben, Cora, and Dev each present exactly once.
Cora presents first. Asha presents before Dev.
Ben presents immediately after Asha.
What is the presentation order?

Solution:
Cora must occupy position 1.
Asha and Ben must be consecutive, with Asha before Ben.
They cannot occupy positions 3 and 4 because Dev must come after Asha.
Therefore, Asha takes position 2 and Ben takes position 3.
Dev takes position 4.
This order satisfies all three rules.
Final answer: Cora, Asha, Ben, Dev.

Example 2:
Question: Hana, Ivan, Jules, and Kiran each present exactly once.
Kiran presents last. Jules presents immediately before Hana.
Ivan presents before Jules.
What is the presentation order?

Solution:
Kiran must occupy position 4.
Jules and Hana must be consecutive, with Jules before Hana.
They cannot occupy positions 1 and 2 because Ivan must come before Jules.
Therefore, Jules takes position 2 and Hana takes position 3.
Ivan takes position 1.
This order satisfies all three rules.
Final answer: Ivan, Jules, Hana, Kiran.

New question:
""".strip()

cot_logic_prompt += (
    "\n" + tasks["logic"]
    + "\n\nEnd your response with: "
    + "Final answer: <names in presentation order, separated by commas>."
)

cot_logic_response = run_prompt(
    technique="chain_of_thought",
    task_name="logic",
    prompt=cot_logic_prompt,
)

log(
    "\nReference answer:",
    ", ".join(reference_answers["logic"]),
)


[2026-09-10T21:24:16+00:00] chain_of_thought | logic | final

PROMPT:
Use the worked examples below as a guide.
For the new question, explain your solution briefly.

Example 1:
Question: Asha, Ben, Cora, and Dev each present exactly once.
Cora presents first. Asha presents before Dev.
Ben presents immediately after Asha.
What is the presentation order?

Solution:
Cora must occupy position 1.
Asha and Ben must be consecutive, with Asha before Ben.
They cannot occupy positions 3 and 4 because Dev must come after Asha.
Therefore, Asha takes position 2 and Ben takes position 3.
Dev takes position 4.
This order satisfies all three rules.
Final answer: Cora, Asha, Ben, Dev.

Example 2:
Question: Hana, Ivan, Jules, and Kiran each present exactly once.
Kiran presents last. Jules presents immediately before Hana.
Ivan presents before Jules.
What is the presentation order?

Solution:
Kiran must occupy position 4.
Jules and Hana must be consecutive, with Jules before Hana.
They cannot occupy pos

## 1.6 Zero-Shot Chain-of-Thought Prompting

Zero-shot Chain-of-Thought prompting asks the model to solve a task
step by step without providing any worked examples.

I will use the same math problem and scheduling puzzle as before.
Each prompt will ask for a brief step-by-step solution followed by
the final answer.

This differs from the previous section because the model will not
receive demonstrations of how to solve similar problems.

I will compare these responses with the ordinary zero-shot responses
and the responses produced using worked examples.

### 1.6.1 Example A — Math Word Problem

I will give the model the original math problem and ask it to solve
the problem step by step.

I will not supply examples or intermediate calculations. I will check
whether the model correctly accounts for each change in the number
of notebooks and calculates the number of complete gift bags.

In [18]:
zero_shot_cot_math_prompt = (
    tasks["math"]
    + "\n\nSolve this problem step by step. Keep the explanation brief."
    + "\nEnd your response with: Final answer: <number> gift bags."
)

zero_shot_cot_math_response = run_prompt(
    technique="zero_shot_cot",
    task_name="math",
    prompt=zero_shot_cot_math_prompt,
)

log("\nReference answer:", reference_answers["math"], "gift bags")


[2026-09-10T21:25:40+00:00] zero_shot_cot | math | final

PROMPT:
A student club buys 5 packs of notebooks, with 12 notebooks in each pack. The club receives 8 additional notebooks as a donation and then gives away 47 notebooks. The remaining notebooks are placed into gift bags containing 3 notebooks each. How many complete gift bags can the club make?

Solve this problem step by step. Keep the explanation brief.
End your response with: Final answer: <number> gift bags.

MODEL RESPONSE:
Let's break down the problem step by step:

1. **Calculate the total number of notebooks initially bought:**
   - The club buys 5 packs of notebooks.
   - Each pack contains 12 notebooks.
   - Total notebooks = 5 packs * 12 notebooks/pack = 60 notebooks.

2. **Add the donation notebooks:**
   - The club receives 8 additional notebooks.
   - New total = 60 notebooks + 8 notebooks = 68 notebooks.

3. **Subtract the notebooks given away:**
   - The club gives away 47 notebooks.
   - New total = 68 noteboo

### 1.6.2 Example B — Scheduling Puzzle

I will ask the model to solve the original scheduling puzzle
step by step without any demonstrations.

I will check whether its proposed order satisfies every rule.
I will also check whether the explanation supports the final order,
rather than assuming that a detailed response must be correct.

In [ ]:
zero_shot_cot_logic_prompt = (
    tasks["logic"]
    + "\n\nSolve this problem step by step. Keep the explanation brief."
    + "\nEnd your response with: "
    + "Final answer: <names in presentation order, separated by commas>."
)

zero_shot_cot_logic_response = run_prompt(
    technique="zero_shot_cot",
    task_name="logic",
    prompt=zero_shot_cot_logic_prompt,
)

log(
    "\nReference answer:",
    ", ".join(reference_answers["logic"]),
)